In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



In [4]:
def fasttext_to_dataframe(file_path):
    labels = []
    texts = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(' ', 1)  # split only first space
            
            label = parts[0].replace('__label__', '')
            text = parts[1] if len(parts) > 1 else ""
            
            labels.append(label)
            texts.append(text)

    df = pd.DataFrame({
        'label': labels,
        'text': texts
    })

    return df

In [5]:
df=fasttext_to_dataframe('test.ft.txt')

In [6]:
y=df['label'].astype(int)
X=df['text']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()

model.add(Embedding(input_dim=10000, output_dim=128, input_length=max_len))

model.add(LSTM(64))

model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
# X_train_pad.dtype
y_train.dtype

dtype('int64')

In [14]:
model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 271s 54ms/step - accuracy: 0.5003 - loss: -82.2419 - val_accuracy: 0.4987 - val_loss: -160.0579
Epoch 2/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 264s 53ms/step - accuracy: 0.5003 - loss: -236.3888 - val_accuracy: 0.4987 - val_loss: -314.3946
Epoch 3/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 330s 54ms/step - accuracy: 0.5003 - loss: -390.1004 - val_accuracy: 0.4987 - val_loss: -468.7203
Epoch 4/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 292s 58ms/step - accuracy: 0.5003 - loss: -543.9244 - val_accuracy: 0.4987 - val_loss: -623.0771
Epoch 5/5
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 271s 54ms/step - accuracy: 0.5003 - loss: -697.8151 - val_accuracy: 0.4987 - val_loss: -777.4141


In [15]:
tricky_reviews = [
   # Strong Positive
    "Absolutely fantastic product exceeded my expectations",
    "Loved it so much will recommend to everyone",
    "Best purchase ever totally worth it",
    
    # Strong Negative
    "Completely useless stopped working in one day",
    "Very disappointed waste of money",
    "Worst experience ever highly not recommended",
    
    # Neutral / Mixed (IMPORTANT 🔥)
    "Product is okay not great but not bad either",
    "Average quality could be better",
    "It works fine but nothing special",
    
    # Tricky Negation Cases (VERY IMPORTANT 🔥🔥)
    "Not bad actually pretty good",
    "Not good at all very disappointing",
    "I do not like this product",
    "I am not unhappy with the purchase",
    
]

In [16]:

y_pred=model.predict(tricky_reviews)

ValueError: Unrecognized data type: x=['Absolutely fantastic product exceeded my expectations', 'Loved it so much will recommend to everyone', 'Best purchase ever totally worth it', 'Completely useless stopped working in one day', 'Very disappointed waste of money', 'Worst experience ever highly not recommended', 'Product is okay not great but not bad either', 'Average quality could be better', 'It works fine but nothing special', 'Not bad actually pretty good', 'Not good at all very disappointing', 'I do not like this product', 'I am not unhappy with the purchase'] (of type <class 'list'>)